<a href="https://colab.research.google.com/github/Layaa-V/M25CSA017-NLU-A2/blob/main/NLU_A2_Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
with open("TrainingNames.txt", "r", encoding="utf-8") as f:
        names = [line.strip().lower() for line in f.readlines() if line.strip()]

In [ ]:
chars = set("".join(names))
chars.add("<SOS>")
chars.add("<EOS>")
chars.add("<PAD>")

char2idx = {c: i for i, c in enumerate(sorted(list(chars)))}
idx2char = {i: c for c, i in char2idx.items()}
VOCAB_SIZE = len(chars)

In [ ]:
def name_to_tensor(name):
    indices = [char2idx["<SOS>"]] + [char2idx[c] for c in name] + [char2idx["<EOS>"]]
    return torch.tensor(indices, dtype=torch.long, device=device)

In [ ]:
class VanillaRNN(nn.Module):
  def __init__(self, vocab_size, embed_dim, hidden_dim):
      super(VanillaRNN, self).__init__()
      self.hidden_dim = hidden_dim
      self.embedding = nn.Embedding(vocab_size, embed_dim)
      self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
      self.fc = nn.Linear(hidden_dim, vocab_size)

  def forward(self, x):
      embedded = self.embedding(x)
      out, hidden = self.rnn(embedded)
      output = self.fc(out)
      return output

In [ ]:
class BLSTM(nn.Module):
  def __init__(self, vocab_size, embed_dim, hidden_dim):
      super(BLSTM, self).__init__()
      self.hidden_dim = hidden_dim
      self.embedding = nn.Embedding(vocab_size, embed_dim)
      self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
      self.fc = nn.Linear(hidden_dim * 2, vocab_size)

  def forward(self, x):
      embedded = self.embedding(x)
      out, (h, c) = self.lstm(embedded)
      output = self.fc(out)
      return output

In [ ]:
class RNNAttention(nn.Module):
  def __init__(self, vocab_size, embed_dim, hidden_dim):
      super(RNNAttention, self).__init__()
      self.hidden_dim = hidden_dim
      self.embedding = nn.Embedding(vocab_size, embed_dim)
      self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
      self.attention_weights = nn.Linear(hidden_dim, 1)
      self.fc = nn.Linear(hidden_dim * 2, vocab_size)

  def forward(self, x):
      embedded = self.embedding(x)
      out, hidden = self.rnn(embedded)

      # Attention scores over all time steps
      attn_scores = self.attention_weights(out)
      attn_weights = torch.softmax(attn_scores, dim=1)   # normalise over seq

      # Weighted context vector, expanded to every time step
      context = torch.sum(attn_weights * out, dim=1, keepdim=True)
      context = context.expand(-1, out.size(1), -1)

      combined = torch.cat((out, context), dim=2)
      output = self.fc(combined)
      return output

In [ ]:
emb_dim  = 64
hid_dim = 128
lr = 0.003
epochs = 50

models = {"Vanilla RNN" : VanillaRNN(VOCAB_SIZE, emb_dim, hid_dim).to(device),
          "BLSTM" : BLSTM(VOCAB_SIZE, emb_dim, hid_dim).to(device),
          "RNN + Attention": RNNAttention(VOCAB_SIZE, emb_dim, hid_dim).to(device),}

In [ ]:
for name, model in models.items():
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name}: {params} trainable parameters")

Vanilla RNN: 30236 trainable parameters
BLSTM: 207644 trainable parameters
RNN + Attention: 33949 trainable parameters


In [ ]:
def model_training(model, names, epochs=epochs):
    criterion = nn.CrossEntropyLoss(ignore_index=char2idx["<PAD>"])
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        shuffled = names.copy()
        np.random.shuffle(shuffled)

        for name in shuffled:
            tensor  = name_to_tensor(name).unsqueeze(0)
            inputs  = tensor[:, :-1]
            targets = tensor[:, 1:]

            optimizer.zero_grad()
            output = model(inputs)
            loss = criterion(output.view(-1, VOCAB_SIZE), targets.view(-1))
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}; loss: {total_loss/len(names):.4f}")
    return model

In [ ]:
for name, model in models.items():
    print(f"{name} : ")
    models[name] = model_training(model, names)

Vanilla RNN : 
  Epoch 10; loss: 1.7796
  Epoch 20; loss: 1.5874
  Epoch 30; loss: 1.3372
  Epoch 40; loss: 1.1660
  Epoch 50; loss: 1.1141
BLSTM : 
  Epoch 10; loss: 0.0000
  Epoch 20; loss: 0.0000
  Epoch 30; loss: 0.0000
  Epoch 40; loss: 0.0000
  Epoch 50; loss: 0.0000
RNN + Attention : 
  Epoch 10; loss: 0.3481
  Epoch 20; loss: 0.2621
  Epoch 30; loss: 0.0802
  Epoch 40; loss: 0.0084
  Epoch 50; loss: 0.0011


In [ ]:
def generate_names(model, num_names=200, max_len=15, temperature=1.1):
    model.eval()
    generated = []

    with torch.no_grad():
        attempts = 0
        while len(generated) < num_names and attempts < num_names * 10:
            attempts += 1
            current_seq = [char2idx["<SOS>"]]

            for step in range(max_len):
                seq_tensor = torch.tensor([current_seq], dtype=torch.long, device=device)
                out        = model(seq_tensor)
                logits     = out[0, -1, :] / temperature

                #forbid EOS on the first step
                if step == 0:
                    logits[char2idx["<EOS>"]] = float('-inf')

                #Repetition penalty to prevent words like "Ccccc" and "Zzz"
                if step > 0:
                    last_char = current_seq[-1]
                    #mildly penalize the exact same character repeating
                    logits[last_char] = logits[last_char] - 1.5

                    #strictly forbid 3 identical characters in a row
                    if step > 1 and current_seq[-1] == current_seq[-2]:
                        logits[last_char] = float('-inf')

                # Nucleus(top-p) sampling
                top_p  = 0.90
                probs  = torch.softmax(logits, dim=0).cpu().numpy()
                sorted_idx   = np.argsort(probs)[::-1]
                cumulative   = np.cumsum(probs[sorted_idx])
                cutoff       = np.searchsorted(cumulative, top_p) + 1
                nucleus_idx  = sorted_idx[:cutoff]
                nucleus_probs = probs[nucleus_idx]
                nucleus_probs = nucleus_probs / nucleus_probs.sum()

                next_char_idx = int(np.random.choice(nucleus_idx, p=nucleus_probs))

                if next_char_idx == char2idx["<EOS>"]:
                    break

                if idx2char[next_char_idx] in ("<PAD>", "<SOS>"):
                    continue

                current_seq.append(next_char_idx)

            #decoding
            name_chars = [
                idx2char[idx] for idx in current_seq[1:]
                if idx2char[idx] not in ("<EOS>", "<PAD>", "<SOS>")
            ]
            name = "".join(name_chars).capitalize()

            #enforce min length to drop junk generations
            if len(name) >= 3:
                generated.append(name)

    return generated[:num_names]

In [ ]:
train_set = set([n.capitalize() for n in names])

for model_name, model in models.items():
    # Generate names
    gen_names = generate_names(model, num_names=200, temperature=1.1)

    #prevent error if model failed to generate valid names
    if len(gen_names) == 0:
        print(f"\n[{model_name}]")
        print(f"  Novelty Rate  : 0.00%")
        print(f"  Diversity     : 0.00%")
        print(f"  Sample Names  : [] (Model collapsed and generated no valid names >= 3 chars)")
        continue #move to next model

    #calculate metrics
    novel_names = [n for n in gen_names if n not in train_set]
    novelty_rate = (len(novel_names) / len(gen_names)) * 100

    unique_names = set(gen_names)
    diversity    = (len(unique_names) / len(gen_names)) * 100

    print(f"\n[{model_name}]")
    print(f"  Novelty Rate  : {novelty_rate:.2f}%")
    print(f"  Diversity     : {diversity:.2f}%")
    print(f"  Sample Names  : {list(unique_names)[:10]}")


[Vanilla RNN]
  Novelty Rate  : 24.50%
  Diversity     : 91.00%
  Sample Names  : ['Vasant', 'Badha', 'Garika', 'Chirag', 'Shith', 'Gita', 'Natasha', 'Swaraswati', 'Palpa', 'Naray']

[BLSTM]
  Novelty Rate  : 100.00%
  Diversity     : 20.45%
  Sample Names  : ['Ffa', 'Ffw', 'Ffb', 'Ffk', 'Ffr', 'Ffy', 'Fft', 'Ffe', 'Ffg']

[RNN + Attention]
  Novelty Rate  : 100.00%
  Diversity     : 4.50%
  Sample Names  : ['Ffxfzfzh', 'Ffxff', 'Ffxch', 'Ffxfzff', 'Ffxfch', 'Ffxfzh', 'Ffxfzf', 'Ffxfzfz', 'Ffxffch']
